# NB04 — Musanga satellite abundance and uncertainty mapping

**CURRENT WORKFLOW: run only Sections 06–09 below.** Sections 00–01 are archived legacy setup and are **not required** for the cached workflow. In particular, do not run the old SSD-staging cell. The experimental Blocks 02–14 remain accessible in Git history; this notebook now prioritizes prediction intervals, ensemble spread and area of applicability.

## 00 — USER SETTINGS

In [ ]:
# ==================== USER SETTINGS ====================
SCALES_M = [10,30,50,75,100,125]
FEATURE_SETS = ['ANNUAL_S2','EMBEDDING','S2_PLUS_EMBEDDING']
MODELS = ['RIDGE','RF','EXTRA_TREES','HIST_GB']
TARGETS = ['mean_probability','cover_fraction_p05']
MAX_SAMPLES_PER_MOSAIC = 8000
MIN_UAV_COVERAGE = .90
SEED = 20261022
COPY_UAV_TO_SSD = True
COPY_FEATURES_TO_SSD = True
FORCE_REBUILD = False  # keep False: resumes cached aggregate/sample/fold files
N_JOBS = 2
SAVE_FIGURES = True
# ========================================================

# BLOCK 06 — targeted high-cover recovery (independent of full Block 04 rerun)
B06_RUN = True
B06_SCALES_M = [75,100,125]
B06_FEATURE_SETS = ['EMBEDDING','S2_PLUS_EMBEDDING']
B06_TARGETS = ['mean_probability','cover_fraction_p05']
B06_WEIGHT_STRENGTHS = [0,2,5]  # 0=standard boosting; >0 emphasizes cover above 20%
B06_HIGH_THRESHOLD = .40
B06_REBUILD = False


# ========== BLOCKS 07–08: ROI seasonal LOSO settings ==========
from pathlib import Path
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
FINAL=ROOT/"Final"
ROI_PATH=ROOT/"data/Orthoo_RGB/ROI.shp"
SCALES=[75,100,125]
TARGETS=["mean_probability"]  # optional: add "cover_fraction_p05"
FEATURE_SETS={
 "EMB":["EMB_ANNUAL"],
 "QUARTERLY":["S2_QUARTERLY_3YR"],
 "EMB_QUARTERLY":["EMB_ANNUAL","S2_QUARTERLY_3YR"],
 "EMB_ANNUAL_QUARTERLY":["EMB_ANNUAL","RAW_ANNUAL","S2_QUARTERLY_3YR"]
}
CV_SCHEMES=["FIVE_MOSAICS","FOUR_FORESTS"]
MAX_CELLS_PER_MOSAIC=8000
MIN_ROI_COVERAGE=0.90    # fractional pixel coverage inside ROI
ROI_SUBPIXELS=4          # 4x4 supersampling
MIN_UAV_COVERAGE=0.90
MIN_QUARTER_CLEAR=1
SEED=20261022
WEIGHT_STRENGTHS=[0,2]
HIGH_COVER_THRESHOLD=0.40
FORCE_SAMPLES=False
FORCE_FOLDS=False
SAVE_PLOTS=True


## 01 — Mount Drive, audit cached inputs and stage them on local SSD

In [ ]:
%pip -q install rasterio geopandas pandas pyarrow scikit-learn matplotlib scipy
import os, json, math, time, hashlib, shutil, gc, warnings
from pathlib import Path
import numpy as np, pandas as pd, rasterio
from affine import Affine
from rasterio.windows import Window
from rasterio.warp import reproject, Resampling
from sklearn.ensemble import RandomForestRegressor,ExtraTreesRegressor,HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt
from IPython.display import display
from google.colab import drive
if not Path('/content/drive/MyDrive').exists():drive.mount('/content/drive')
ROOT=next((p for p in [Path('/content/drive/MyDrive/Colab Notebooks/Musanga/Final'),Path('/content/drive/MyDrive/Musanga/Final')] if p.is_dir()),None)
assert ROOT is not None
OLD=ROOT/'Outputs/V3/04_satellite_Musanga/features'
MAPS=ROOT/'Outputs/V3_1/03_M2_warmstart_c6a33504c37efb2ebb84_full_mosaic/probability_maps'
OUT=ROOT/'Outputs/V3_1/04_satellite_full_warmstart'
CACHE=OUT/'cache';AGG=CACHE/'aggregates';SAMPLE=CACHE/'samples';FOLDS=OUT/'folds';FIG=OUT/'figures'
for p in (CACHE,AGG,SAMPLE,FOLDS,FIG):p.mkdir(parents=True,exist_ok=True)
SSD=Path('/content/musanga_nb04_ssd');(SSD/'uav').mkdir(parents=True,exist_ok=True);(SSD/'features').mkdir(exist_ok=True)
YEARS={'Baego':2023,'MOS':2024,'Yangambi':2024,'Yoko':2020}
SITE_MOSAICS={'Baego':['Baego.tif'],
 'MOS':['Mos_1_2_transparent_mosaic_group1_georef.tif','Mos_5_6_transparent_mosaic_group1_georef.tif'],
 'Yangambi':['Yangambi_georef.tif'],'Yoko':['Yoko_kvo_georef.tif']}
FAMILY_NAMES={'ANNUAL_S2':['RAW_ANNUAL'],'EMBEDDING':['EMB_ANNUAL'],
              'S2_PLUS_EMBEDDING':['RAW_ANNUAL','EMB_ANNUAL']}
def uavfile(name):return MAPS/f'{Path(name).stem}_M2_warmstart_epoch02_probability_u16_FULL.tif'
def featfile(site,fam):return OLD/fam/f'{site}_{fam}_{YEARS[site]}_10m.tif'
def staged(path,folder,copy):
    assert path.is_file(),('Missing required cached file',path)
    target=folder/path.name
    if not copy:return path
    srcstat=path.stat()
    if not target.exists() or target.stat().st_size!=srcstat.st_size:
        print('COPY to SSD:',path.name,round(srcstat.st_size/1e9,2),'GB',flush=True)
        tmp=target.with_name(target.name+'.partial')
        shutil.copyfile(path,tmp);os.replace(tmp,target)
    return target
UAV={name:staged(uavfile(name),SSD/'uav',COPY_UAV_TO_SSD) for ns in SITE_MOSAICS.values() for name in ns}
FAMILY={'RAW_ANNUAL','EMB_ANNUAL'}
assert all(featfile(site,fam).is_file() for site in SITE_MOSAICS for fam in FAMILY)
FEATURES={(site,fam):staged(featfile(site,fam),SSD/'features',COPY_FEATURES_TO_SSD)
          for site in SITE_MOSAICS for fam in sorted(FAMILY)}
print('Input audit: ',len(UAV),'UAV maps,',len(FEATURES),'cached predictor stacks')
print('SSD free GB:',round(shutil.disk_usage(SSD).free/1e9,2),flush=True)


## Archived legacy setup ends here

The following independent mapping workflow uses existing `roi_cells_100m.parquet` / `roi_cells_75m.parquet` and aligned annual satellite raster grids. Do not run the legacy staging cells above.

## 06 — One configuration, no extraction

Run 06 → 09 only. Default: 100 m, annual embeddings + quarterly Sentinel-2. Balanced capped training prevents MOS from dominating; globally summarized geographic validation averages four forests equally. No legacy blocks are required.

In [ ]:
# 06 — INPUTS, settings, forest-balanced sampling & spatial partitions
# Run ONLY sections 06 onward. No raster extraction or legacy setup needed.
from pathlib import Path
import numpy as np, pandas as pd, rasterio, matplotlib.pyplot as plt, json
from scipy.spatial import cKDTree
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import r2_score
from IPython.display import display
from google.colab import drive
if not Path("/content/drive/MyDrive").is_dir():drive.mount("/content/drive")
ROOT=Path("/content/drive/MyDrive/Colab Notebooks/Musanga")
CACHE=ROOT/"Final/Outputs/V3_1/04B_roi_quarterly_LOSO/samples"
SAT=ROOT/"Final/Outputs/V3/04_satellite_Musanga/features/RAW_ANNUAL"
OUT=ROOT/"Final/Outputs/V3_1/04_uncertainty_mapping"
OUT.mkdir(parents=True,exist_ok=True)
# Settings: change here; validation and map stages use the same values.
SCALE_M=100  # recommended; optionally rerun at 75 m for sensitivity
YEARS={"Baego":2023,"MOS":2024,"Yangambi":2024,"Yoko":2020}
FORESTS=["Baego","MOS","Yangambi","Yoko"]
FEATURE_FAMILIES=["EMB_ANNUAL","S2_QUARTERLY_3YR"]
N_ENSEMBLE=5
TRAIN_CAP_PER_FOREST=500  # equal forest contribution; MOS cannot dominate by size
TRAIN_STRATEGY='stratified'  # baseline='random'; stratify ONLY within training partition
ABUNDANCE_EDGES=[0,.10,.20,.40,1.000001]
MIN_HIGH_SHARE=.25  # target high >=40% where available; no synthetic oversampling
BLOCK_SIZE_M=500
SPATIAL_BUFFER_M=150
TEST_FRACTION=.20
CAL_FRACTION=.20
INTERVAL_LEVEL=.90
REPEATS=3
SEED=20261022
FORCE_VALIDATION=False
# Required artifact, aligned original Block 07 satellite/UAV sampling
SAMPLE_PATH=CACHE/f"roi_cells_{SCALE_M}m.parquet"
assert SAMPLE_PATH.is_file(),f"Cached data missing: {SAMPLE_PATH}"
all_cells=pd.read_parquet(SAMPLE_PATH).reset_index(drop=True)
assert set(all_cells.site.unique())==set(FORESTS),all_cells.site.unique()
assert {"site","group","mosaic","row","col","mean_probability"}.issubset(all_cells.columns)
FEATURE_COLS=[c for c in all_cells if any(c.startswith(f+"__") for f in FEATURE_FAMILIES)]
assert len(FEATURE_COLS)>60,"Missing embedding/seasonal feature columns"
assert all_cells[FEATURE_COLS].notna().all().all()
assert all_cells.mean_probability.between(0,1).all()
assert not all_cells.duplicated(["site","group","row","col"]).any(),"Duplicate spatial samples"
# A cell's row/col are its position on that forest's satellite grid (not geodetic coordinates).
# Group-prefixed spatial blocks prevent falsely merging the two MOS mosaics.
df=all_cells.copy()
df["x_m"]=(df.col.to_numpy(float)+.5)*SCALE_M
df["y_m"]=(df.row.to_numpy(float)+.5)*SCALE_M
df["block_id"]=(df.group.astype(str)+"_"+
    np.floor(df.x_m/BLOCK_SIZE_M).astype(int).astype(str)+"_"+
    np.floor(df.y_m/BLOCK_SIZE_M).astype(int).astype(str))
print("CACHED SPATIAL SUPPORT:",len(df),"cells at",SCALE_M,"m")
display(df.groupby("site").mean_probability.agg(
    n="size",mean_cover="mean",p90=lambda s:s.quantile(.9),
    high_fraction=lambda s:(s>=.4).mean()).round(3))
print("Full-region feature rasters beyond the UAV footprints are NOT available in this cache.")
def subset_cap(frame,max_n,rng):
    if len(frame)<=max_n:return frame.copy()
    return frame.iloc[np.sort(rng.choice(len(frame),max_n,replace=False))].copy()
def stratified_cap(frame,max_n,rng):
    # Oversample the *selection share* of high covers, never duplicate cells;
    # all sampling happens after geographic train/cal/test splitting.
    if len(frame)<=max_n:return frame.copy()
    x=frame.copy()
    x["_abundance_bin"]=pd.cut(x.mean_probability,ABUNDANCE_EDGES,
                               labels=False,include_lowest=True,right=False)
    pool={int(k):group.index.to_numpy() for k,group in x.groupby("_abundance_bin")}
    chosen=[];high=pool.get(3,np.array([],dtype=int))
    n_high=min(len(high),int(np.ceil(max_n*MIN_HIGH_SHARE)))
    if n_high:chosen.extend(rng.choice(high,size=n_high,replace=False).tolist())
    remaining=max_n-len(chosen)
    # Quotas for remaining bins are distributed evenly among available classes.
    leftovers={k:np.setdiff1d(ids,np.array(chosen,dtype=int)) for k,ids in pool.items()}
    while remaining>0:
        active=[k for k,ids in leftovers.items() if len(ids)>0]
        if not active:break
        n_each=max(1,remaining//len(active))
        progress=0
        for k in active:
            m=min(n_each,len(leftovers[k]),remaining)
            if not m:continue
            picked=rng.choice(leftovers[k],size=m,replace=False)
            chosen.extend(picked.tolist())
            leftovers[k]=np.setdiff1d(leftovers[k],picked)
            remaining-=m;progress+=m
            if remaining==0:break
        if progress==0:break
    assert len(chosen)==max_n,(len(chosen),max_n)
    return frame.loc[chosen].copy()
def balanced_training(frame,rng):
    portions=[]
    for site in FORESTS:
        part=frame[frame.site==site]
        if TRAIN_STRATEGY=="stratified":
            d=stratified_cap(part,TRAIN_CAP_PER_FOREST,rng)
        elif TRAIN_STRATEGY=="random":
            d=subset_cap(part,TRAIN_CAP_PER_FOREST,rng)
        else:raise ValueError(TRAIN_STRATEGY)
        if len(d)<30:raise ValueError(f"Too few training rows for {site}: {len(d)}")
        portions.append(d)
    return pd.concat(portions,ignore_index=True)
def split_spatial(frame,repeat):
    rng=np.random.default_rng(SEED+SCALE_M*100+repeat)
    tr_parts=[];ca_parts=[];te_parts=[]
    for site in FORESTS:
        whole=frame[frame.site==site].copy()
        groups=whole.groupby("block_id").indices
        keys=list(groups);rng.shuffle(keys)
        held={"test":[],"cal":[]};cursor=0
        for split,fraction in [("test",TEST_FRACTION),("cal",CAL_FRACTION)]:
            count=0
            while cursor<len(keys) and count<fraction*len(whole):
                key=keys[cursor];cursor+=1
                held[split].append(key);count+=len(groups[key])
        a=whole[~whole.block_id.isin(held["test"]+held["cal"])].copy()
        b=whole[whole.block_id.isin(held["cal"])].copy()
        c=whole[whole.block_id.isin(held["test"])].copy()
        # Real-world distances only within the same forest footprint.
        def remove_near(candidate,reference):
            if candidate.empty or reference.empty:return candidate.iloc[0:0].copy()
            nearest=cKDTree(reference[["x_m","y_m"]].to_numpy()).query(
                candidate[["x_m","y_m"]].to_numpy(),k=1)[0]
            return candidate.iloc[np.flatnonzero(nearest>=SPATIAL_BUFFER_M)].copy()
        b=remove_near(b,a);c=remove_near(c,a);c=remove_near(c,b)
        if min(len(a),len(b),len(c))<8:
            raise ValueError(("Insufficient independent blocks",site,repeat,
                              len(a),len(b),len(c),"Try smaller buffer or larger coverage"))
        tr_parts.append(a);ca_parts.append(b);te_parts.append(c)
    return tuple(pd.concat(items,ignore_index=True) for items in (tr_parts,ca_parts,te_parts))
def fit_ensemble(train,predict_sets,seed):
    models=[];rng=np.random.default_rng(seed)
    for i in range(N_ENSEMBLE):
        fit=balanced_training(train,np.random.default_rng(seed+i*113))
        # Ensemble varies training draws within each forest; fixed boosting settings.
        model=HistGradientBoostingRegressor(learning_rate=.06,max_iter=200,
            max_leaf_nodes=20,min_samples_leaf=40,l2_regularization=2,
            early_stopping=False,random_state=seed+i)
        model.fit(fit[FEATURE_COLS].to_numpy(np.float32),
                  fit.mean_probability.to_numpy(np.float32),
                  sample_weight=fit.site.map(fit.site.value_counts().rdiv(1)).to_numpy(float)
                    / np.mean(fit.site.map(fit.site.value_counts().rdiv(1)).to_numpy(float)))
        models.append(model)
    outputs=[]
    for group in predict_sets:
        x=group[FEATURE_COLS].to_numpy(np.float32)
        arr=np.stack([np.clip(m.predict(x),0,1) for m in models],axis=0)
        outputs.append((arr.mean(axis=0),arr.std(axis=0)))
    return models,outputs


## 07 — Spatial validation plus calibration-only refinement

Run once after Section 06. The archived **stratified v3 predictor is unchanged**; the appended refinement reconstructs identical calibration predictions on the same spatial folds and then recalculates intervals using **all spatial calibration cells**, each forest receiving equal total weight. Compare old and new interval coverage and width on exactly matched test cells. Point-prediction results stay constant. Files have distinct `weightedcal_v4` names.

In [ ]:
# 07 — SPATIALLY RESERVED TRAIN / CALIBRATION / TEST & global, equal-forest metrics
VALID_DIR=OUT/"validation";VALID_DIR.mkdir(exist_ok=True)
def empirical_quantile(values,level):
    v=np.sort(np.asarray(values,float))
    k=min(len(v),int(np.ceil((len(v)+1)*level)))
    return float(v[k-1])
def forest_balanced_calibration(cal,pred):
    # Calibrate *signed* residual tails. Equal-forest subsample prevents MOS
    # setting the error distribution; predicted bins make interval shape adaptive.
    # These are empirical, not coverage-guaranteed under spatial/domain shift.
    rng=np.random.default_rng(SEED+824)
    n_each=int(cal.site.value_counts().min())
    idx=np.concatenate([rng.choice(np.flatnonzero(cal.site.to_numpy()==forest),
                       n_each,replace=False) for forest in FORESTS])
    error=cal.mean_probability.to_numpy(float)[idx]-np.asarray(pred,float)[idx]
    fitted=np.asarray(pred,float)[idx]
    alpha=1-INTERVAL_LEVEL
    def signed_limits(v):
        v=np.sort(np.asarray(v,float));n=len(v)
        # Conservative signed-residual empirical quantiles.
        lo=v[max(0,int(np.floor((n+1)*alpha/2))-1)]
        hi=v[min(n-1,int(np.ceil((n+1)*(1-alpha/2)))-1)]
        return [float(lo),float(hi)]
    global_limits=signed_limits(error)
    # Bins depend ONLY on predicted fractions; sparse strata use pooled limits.
    edges=[0,.10,.25,1.000001]
    bins=[]
    for a,b in zip(edges[:-1],edges[1:]):
        within=(fitted>=a)&(fitted<b)
        n=int(within.sum())
        limits=signed_limits(error[within]) if n>=25 else global_limits
        bins.append(dict(min_pred=a,max_pred=b,n_cal=n,limits=limits,
                         pooled_fallback=n<25))
    return dict(n_per_forest=n_each,n_total=len(idx),global_limits=global_limits,
                bins=bins,bin_edges=edges)
def interval_bounds(pred,calibration):
    pred=np.asarray(pred,float)
    low=np.empty_like(pred);high=np.empty_like(pred)
    for segment in calibration["bins"]:
        mask=(pred>=segment["min_pred"])&(pred<segment["max_pred"])
        lo,hi=segment["limits"]
        low[mask]=np.clip(pred[mask]+lo,0,1)
        high[mask]=np.clip(pred[mask]+hi,0,1)
    assert np.all(high>=low),"Asymmetric interval boundaries invalid"
    return low,high
def score(sub):
    y=sub.observed.to_numpy(float);p=sub.predicted.to_numpy(float)
    high=y>=.4
    return dict(n=len(sub),R2=r2_score(y,p) if len(y)>1 and y.std()>1e-8 else np.nan,
                MAE_pp=100*np.mean(abs(y-p)),
                RMSE_pp=100*np.sqrt(np.mean((y-p)**2)),
                bias_pp=100*np.mean(p-y),
                PI90_coverage=np.mean((y>=sub.lower90)&(y<=sub.upper90)),
                PI90_width_pp=100*np.mean(sub.upper90-sub.lower90),
                high_n=int(high.sum()),
                high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,
                high_PI90_coverage=np.mean(
                    (y[high]>=sub.lower90.to_numpy()[high])&
                    (y[high]<=sub.upper90.to_numpy()[high])) if high.any() else np.nan,
                ensemble_sd_pp=100*np.mean(sub.ensemble_sd))
def novelty_fit(tr,query):
    # PCA nearest-neighbour support check: exploratory, not a validated AOA guarantee.
    trX=tr[FEATURE_COLS].to_numpy(np.float32)
    qX=query[FEATURE_COLS].to_numpy(np.float32)
    scaler=StandardScaler().fit(trX)
    z=scaler.transform(trX);q=scaler.transform(qX)
    pca=PCA(n_components=min(10,z.shape[1],len(z)-1),random_state=SEED).fit(z)
    z=pca.transform(z);q=pca.transform(q)
    nn=NearestNeighbors(n_neighbors=2).fit(z)
    training_distance=nn.kneighbors(z)[0][:,1]
    test_distance=nn.kneighbors(q,n_neighbors=1)[0][:,0]
    limit=float(np.quantile(training_distance,.95))
    return test_distance,limit
all_results=[];all_predictions=[];all_cal=[]
for repeat in range(REPEATS):
    cache=VALID_DIR/f"heldout_stratified_v3_{SCALE_M}m_rep{repeat}.parquet"
    calibration_meta=VALID_DIR/f"calibration_stratified_v3_{SCALE_M}m_rep{repeat}.json"
    if cache.exists() and calibration_meta.exists() and not FORCE_VALIDATION:
        test_out=pd.read_parquet(cache)
        info=json.loads(calibration_meta.read_text())
        print("CACHED VALIDATION",repeat,flush=True)
    else:
        train,cal,test=split_spatial(df,repeat)
        models,preds=fit_ensemble(train,[cal,test],SEED+repeat*193)
        cal_mean,_=preds[0];test_mean,test_sd=preds[1]
        calibration=forest_balanced_calibration(cal,cal_mean)
        lower,upper=interval_bounds(test_mean,calibration)
        dist,limit=novelty_fit(train,test)
        test_out=test[["site","group","row","col"]].copy()
        test_out["observed"]=test.mean_probability.to_numpy(float)
        test_out["predicted"]=test_mean
        test_out["lower90"]=lower
        test_out["upper90"]=upper
        test_out["ensemble_sd"]=test_sd
        test_out["novelty_distance"]=dist
        test_out["novel"]=dist>limit
        info=dict(repeat=repeat,n_train=len(train),n_cal=len(cal),n_test=len(test),
                  n_cal_each_forest=calibration["n_per_forest"],
                  strategy=TRAIN_STRATEGY,training_cap=TRAIN_CAP_PER_FOREST,
                  calibration=calibration,novelty_limit=limit)
        test_out.to_parquet(cache,index=False)
        calibration_meta.write_text(json.dumps(info,indent=2))
        print("VALIDATED",repeat,info,flush=True)
    all_predictions.append(test_out)
    all_cal.append(info)
    for forest,chunk in test_out.groupby("site"):
        all_results.append(dict(repeat=repeat,forest=forest,**score(chunk)))
metrics=pd.DataFrame(all_results)
metrics.to_csv(VALID_DIR/f"per_forest_audit_stratified_v3_{SCALE_M}m.csv",index=False)
# GLOBAL metrics ONLY; equal weight to forests and equal weight to spatial repetitions.
numerical=["MAE_pp","RMSE_pp","bias_pp","PI90_coverage","PI90_width_pp",
           "high_PI90_coverage","ensemble_sd_pp"]
summary=metrics.groupby("repeat")[numerical].mean(numeric_only=True)
# R² needs a global pooled definition, not averaging per-forest R².
for rep,pred in enumerate(all_predictions):
    y=pred.observed.to_numpy(float);p=pred.predicted.to_numpy(float)
    forest=pred.site.to_numpy()
    w=np.array([1/(len(FORESTS)*np.sum(forest==s)) for s in forest])
    mu=np.dot(w,y);total=np.dot(w,(y-mu)**2)
    summary.loc[rep,"balanced_global_R2"]=1-np.dot(w,(y-p)**2)/total
    summary.loc[rep,"n_test"]=len(pred)
    summary.loc[rep,"n_test_high"]=int(np.sum(y>=.4))
summary.to_csv(VALID_DIR/f"global_metrics_by_repeat_stratified_v3_{SCALE_M}m.csv")
global_means=summary.mean(numeric_only=True).to_frame("mean").join(
    summary.std(numeric_only=True).to_frame("sd"))
global_means.to_csv(VALID_DIR/f"global_metrics_stratified_v3_{SCALE_M}m.csv")
print("GLOBAL EQUAL-FOREST SPATIAL VALIDATION (mean and SD across repeats)")
display(global_means.round(3))
print("Calibration metadata:",all_cal)
print("Spatial test predictions and per-forest audits saved; global metrics are primary.")
# Compare EXACT same held-out spatial cells with archived random-training baseline.
baseline_rows=[];matched_rows=[]
for rep,new in enumerate(all_predictions):
    original=VALID_DIR/f"heldout_asymmetric_v2_{SCALE_M}m_rep{rep}.parquet"
    if not original.exists():
        print("BASELINE MISSING",original,flush=True);continue
    old=pd.read_parquet(original)
    keys=["site","group","row","col"]
    common=new.merge(old,on=keys,suffixes=("_new","_baseline"),validate="one_to_one")
    assert len(common)==len(new)==len(old),(
        "Different holdout cells; cannot claim paired comparison",rep,len(common),len(new),len(old))
    assert np.allclose(common.observed_new,common.observed_baseline,atol=1e-7)
    for label in ["new","baseline"]:
        y=common[f"observed_{label}"].to_numpy(float)
        p=common[f"predicted_{label}"].to_numpy(float)
        forests=common.site.to_numpy()
        weights=np.array([1/(len(FORESTS)*np.sum(forests==site)) for site in forests])
        mu=np.dot(weights,y)
        high=y>=.4
        covered=(y>=common[f"lower90_{label}"].to_numpy(float))&(
                 y<=common[f"upper90_{label}"].to_numpy(float))
        baseline_rows.append(dict(repeat=rep,model=label,n_test=len(y),n_high=int(high.sum()),
          balanced_R2=1-np.dot(weights,(y-p)**2)/np.dot(weights,(y-mu)**2),
          balanced_MAE_pp=sum(100*np.mean(abs((y-p)[forests==site])) for site in FORESTS)/len(FORESTS),
          high_MAE_pp=100*np.mean(abs((y-p)[high])) if high.any() else np.nan,
          high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,
          high_PI90_coverage=float(np.mean(covered[high])) if high.any() else np.nan,
          overall_PI90_coverage=sum(np.mean(covered[forests==site]) for site in FORESTS)/len(FORESTS)))
if baseline_rows:
    comparison=pd.DataFrame(baseline_rows)
    comparison.to_csv(VALID_DIR/f"paired_random_vs_stratified_{SCALE_M}m.csv",index=False)
    print("\\n=== PAIRED PERFORMANCE: same test cells, random vs high-cover stratified training ===")
    display(comparison.groupby("model").mean(numeric_only=True).round(3))
else:
    print("No archived v2 baseline found; new metrics saved independently.")


# === 07B — REFINE INTERVALS ONLY; freeze saved v3 point predictions ===
# Fit the IDENTICAL stratified ensembles to reconstruct calibration predictions,
# but always retain the archived v3 point predictions / ensemble SD on test rows.
# Full calibration observations are used with equal total weight per forest.
def weighted_tail(values,weights,q):
    order=np.argsort(values)
    x=np.asarray(values,float)[order]
    w=np.asarray(weights,float)[order]
    return float(x[min(len(x)-1,np.searchsorted(np.cumsum(w)/w.sum(),q,side="left"))])
def weighted_effective_n(w):
    w=np.asarray(w,float)
    return float((w.sum()**2)/np.sum(w**2))
def weighted_signed_calibration(cal,pred):
    y=cal.mean_probability.to_numpy(float)
    p=np.asarray(pred,float)
    forest=cal.site.to_numpy()
    weights=np.array([1/(len(FORESTS)*np.sum(forest==site)) for site in forest])
    error=y-p
    alpha=1-INTERVAL_LEVEL
    global_limits=[weighted_tail(error,weights,alpha/2),
                   weighted_tail(error,weights,1-alpha/2)]
    segments=[]
    # Continuously shrink bin-specific quantiles towards pooled limits,
    # avoiding the prior hard switch at 25 equal-subsampled observations.
    for lo,hi in [(0,.10),(.10,.25),(.25,1.000001)]:
        mask=(p>=lo)&(p<hi)
        count=int(mask.sum())
        if count>=10:
            local=[weighted_tail(error[mask],weights[mask],alpha/2),
                   weighted_tail(error[mask],weights[mask],1-alpha/2)]
            eff_n=weighted_effective_n(weights[mask])
            strength=min(.75,eff_n/(eff_n+40))
        else:
            local=global_limits
            eff_n=weighted_effective_n(weights[mask]) if count else 0
            strength=0.
        bounds=((1-strength)*np.asarray(global_limits)+
                strength*np.asarray(local)).tolist()
        segments.append(dict(min_pred=lo,max_pred=hi,n_cal=count,
                weighted_effective_n=round(eff_n,2),local_weight=strength,
                limits=bounds,raw_local_limits=local))
    return dict(n_cal=len(y),n_cal_by_forest={site:int((forest==site).sum())
                for site in FORESTS},global_limits=global_limits,bins=segments)
refined_rows=[];refined_predictions=[];refined_cal=[]
for repeat in range(REPEATS):
    old=all_predictions[repeat].copy()
    save=VALID_DIR/f"heldout_weightedcal_v4_{SCALE_M}m_rep{repeat}.parquet"
    meta=VALID_DIR/f"calibration_weightedcal_v4_{SCALE_M}m_rep{repeat}.json"
    if save.exists() and meta.exists() and not FORCE_VALIDATION:
        refreshed=pd.read_parquet(save)
        info=json.loads(meta.read_text())
        print("CACHED WEIGHTED CALIBRATION",repeat,flush=True)
    else:
        train,cal,test=split_spatial(df,repeat)
        _,predictions=fit_ensemble(train,[cal,test],SEED+repeat*193)
        cal_prediction,_=predictions[0]
        test_prediction,test_sd=predictions[1]
        keys=["site","group","row","col"]
        check=test[keys].copy()
        check["predicted"]=test_prediction
        check["observed"]=test.mean_probability.to_numpy(float)
        matched=old.merge(check,on=keys,suffixes=("_old","_recomputed"),
                          validate="one_to_one",sort=False)
        assert len(matched)==len(old)==len(test),"Geographic split has changed"
        assert np.allclose(matched.predicted_old,matched.predicted_recomputed,atol=2e-6),(
            "Point predictions have changed; calibration-only comparison invalid")
        assert np.allclose(matched.observed_old,matched.observed_recomputed,atol=1e-7)
        calibration=weighted_signed_calibration(cal,cal_prediction)
        refreshed=old.copy()
        refreshed["lower90"],refreshed["upper90"]=interval_bounds(
            refreshed.predicted.to_numpy(float),calibration)
        info=dict(repeat=repeat,n_train=len(train),n_cal=len(cal),
                  n_test=len(test),calibration=calibration,
                  source_point_predictions="heldout_stratified_v3",model_unchanged=True)
        refreshed.to_parquet(save,index=False)
        meta.write_text(json.dumps(info,indent=2))
        print("WEIGHTED CALIBRATION",repeat,"n_cal",len(cal),
              "weighted effective support",
              [round(i["weighted_effective_n"],1) for i in calibration["bins"]],
              flush=True)
    assert len(refreshed)==len(old) and np.allclose(
        refreshed.predicted,old.predicted,atol=1e-7)
    refined_predictions.append(refreshed)
    refined_cal.append(info)
    for model,sub in [("previous_v3",old),("weighted_v4",refreshed)]:
        y=sub.observed.to_numpy(float);p=sub.predicted.to_numpy(float)
        forest=sub.site.to_numpy()
        weights=np.array([1/(len(FORESTS)*np.sum(forest==site)) for site in forest])
        covered=(y>=sub.lower90.to_numpy(float))&(y<=sub.upper90.to_numpy(float))
        high=y>=.4
        row=dict(repeat=repeat,model=model,n_test=len(sub),n_high=int(high.sum()),
            balanced_global_coverage=sum(np.mean(covered[forest==site])
                              for site in FORESTS)/len(FORESTS),
            high_cover_coverage=np.mean(covered[high]) if high.any() else np.nan,
            interval_width_pp=100*sum(np.mean(
                (sub.upper90.to_numpy(float)-sub.lower90.to_numpy(float))[forest==site])
                for site in FORESTS)/len(FORESTS),
            balanced_MAE_pp=100*sum(np.mean(abs((y-p)[forest==site]))
                for site in FORESTS)/len(FORESTS),
            high_bias_pp=100*np.mean((p-y)[high]) if high.any() else np.nan,
            novelty_coverage_inside=np.mean(covered[~sub.novel.to_numpy(bool)])
                if (~sub.novel.to_numpy(bool)).any() else np.nan,
            novelty_coverage_outside=np.mean(covered[sub.novel.to_numpy(bool)])
                if sub.novel.to_numpy(bool).any() else np.nan,
            novelty_MAE_inside_pp=100*np.mean(abs((y-p)[~sub.novel.to_numpy(bool)]))
                if (~sub.novel.to_numpy(bool)).any() else np.nan,
            novelty_MAE_outside_pp=100*np.mean(abs((y-p)[sub.novel.to_numpy(bool)]))
                if sub.novel.to_numpy(bool).any() else np.nan)
        refined_rows.append(row)
refined_compare=pd.DataFrame(refined_rows)
refined_compare.to_csv(VALID_DIR/f"paired_interval_calibration_v3_v4_{SCALE_M}m.csv",index=False)
print("\n=== IDENTICAL PREDICTIONS: OLD VS ALL-CALIBRATION WEIGHTED INTERVALS ===",flush=True)
display(refined_compare.groupby("model").mean(numeric_only=True).round(3))
print("\n=== PREDICTION INTERVAL COVERAGE BY OBSERVED ABUNDANCE (audit only) ===")
abundance_rows=[]
for repeat,old in enumerate(all_predictions):
    for model,sub in [("previous_v3",old),("weighted_v4",refined_predictions[repeat])]:
        y=sub.observed.to_numpy(float)
        in_interval=(y>=sub.lower90.to_numpy(float))&(
                     y<=sub.upper90.to_numpy(float))
        for label,mask in [("<20%",y<.20),("20-40%",(y>=.20)&(y<.40)),
                           (">=40%",y>=.40)]:
            if mask.any():
                abundance_rows.append(dict(repeat=repeat,model=model,
                    observed_class=label,n=int(mask.sum()),
                    coverage=float(np.mean(in_interval[mask]))))
abundance_comp=pd.DataFrame(abundance_rows)
abundance_comp.to_csv(VALID_DIR/f"paired_abundance_coverage_v3_v4_{SCALE_M}m.csv",index=False)
display(abundance_comp.groupby(["model","observed_class"]).apply(
    lambda g:pd.Series(dict(n_total=g.n.sum(),
         pooled_coverage=np.average(g.coverage,weights=g.n))),include_groups=False).round(3))
# Export stage consumes the latest weighted calibration only, after coverage review.
mapping_cal=refined_cal
print("Refinement done. Hold Block 09 until coverage is assessed.",flush=True)


## 08 — Global uncertainty diagnostics

The Section 07 output now includes a paired v3 versus v4 interval comparison and a separate novelty-domain audit. Section 08 can be used for the original global scatterplots, but **its default `all_predictions` remain the v3 intervals**; the new v4 coverage comparison printed by Section 07 is authoritative for the refinement.

In [ ]:
# 08 — Ensemble, interval and predictor-domain diagnostics (cached held-out predictions)
from IPython.display import display
P=pd.concat(all_predictions,ignore_index=True)
P["error_pp"]=100*(P.predicted-P.observed)
P["observed_class"]=pd.cut(P.observed,[-.001,.2,.4,1.001],
                            labels=["<20%","20–40%","≥40%"])
bins=P.groupby("observed_class",observed=True).agg(
    n=("observed","size"),MAE_pp=("error_pp",lambda v:abs(v).mean()),
    bias_pp=("error_pp","mean"),
    PI90_coverage=("observed",lambda s:np.nan))
# Calculate coverage from the original frame, not from separate columns in .agg.
coverage=P.assign(covered=(P.observed>=P.lower90)&(P.observed<=P.upper90))
bins["PI90_coverage"]=coverage.groupby("observed_class",observed=True).covered.mean()
bins.to_csv(OUT/f"uncertainty_by_abundance_{SCALE_M}m.csv")
print("GLOBAL diagnostic only; repeated test sets are not independent replicates")
display(bins.round(3))
fig,axes=plt.subplots(1,3,figsize=(13,3.8),constrained_layout=True)
axes[0].scatter(P.observed*100,P.predicted*100,s=6,alpha=.15,rasterized=True)
axes[0].plot([0,100],[0,100],"--",color="grey")
axes[0].set(xlim=(0,100),ylim=(0,100),xlabel="UAV target (%)",
            ylabel="Predicted Musanga (%)",title="Spatially held-out predictions")
axes[1].hist(P.ensemble_sd*100,bins=30)
axes[1].set(xlabel="Ensemble SD (percentage points)",title="Model disagreement")
axes[2].scatter(P.predicted*100,100*(P.upper90-P.lower90),
                s=6,alpha=.2,rasterized=True)
axes[2].set(xlabel="Predicted Musanga (%)",ylabel="90% interval width (pp)",
            title="Empirically calibrated interval width")
fig.savefig(OUT/f"global_diagnostics_{SCALE_M}m.png",dpi=175)
plt.show();plt.close(fig)


## 09 — Full satellite prediction: orthomosaic footprints + 5 km buffer

Run **09A** to derive year-specific mapping footprints from cached UAV mosaics, **09B** to export year-matched satellite predictors across each buffered footprint via Earth Engine, **09C** to audit feature compatibility and produce the six-band maps. Each stage caches outputs. The map footprint follows the valid orthomosaic coverage plus a 5 km buffer (including the UAV interior). MOS 1–2 and 5–6 form a single mapped MOS domain. Separate maps retain their original 2020/2023/2024 reference years: not a contemporaneous regional snapshot. Extrapolation beyond the UAV footprints requires strict applicability flags and intervals are provisional, not guaranteed 90% in unsampled areas.

In [ ]:
# 09A — Derive footprint + 5 km buffers and 100 m canonical satellite grids
# Uses cached 10 m UAV aggregates only; no Earth Engine calls.
from pathlib import Path
import math,json,numpy as np,pandas as pd,rasterio
from rasterio.features import shapes
from rasterio.transform import Affine
from rasterio.warp import reproject,Resampling
from shapely.geometry import shape,mapping
from shapely.ops import unary_union
from IPython.display import display
MAP_ROOT=OUT/f"maps_buffer5km_{SCALE_M}m"
MAP_ROOT.mkdir(parents=True,exist_ok=True)
BUFFER_DISTANCE_M=5000
SOURCE_AGG=ROOT/"Final/Outputs/V3_1/04_satellite_full_warmstart/cache/aggregates"
MOSAIC_FILES={"Baego":["Baego.tif"],
              "MOS":["Mos_1_2_transparent_mosaic_group1_georef.tif",
                     "Mos_5_6_transparent_mosaic_group1_georef.tif"],
              "Yangambi":["Yangambi_georef.tif"],
              "Yoko":["Yoko_kvo_georef.tif"]}
REGIONS={}
for site,names in MOSAIC_FILES.items():
    annual=SAT/f"{site}_RAW_ANNUAL_{YEARS[site]}_10m.tif"
    if not annual.is_file():raise FileNotFoundError(annual)
    with rasterio.open(annual) as ref:
        assert ref.crs and ref.crs.is_projected,("Must buffer in metres",site,ref.crs)
        coarse_transform=ref.transform*Affine.scale(SCALE_M/10,SCALE_M/10)
        standard_origin=(float(coarse_transform.c),float(coarse_transform.f))
        crs=ref.crs
    hulls=[]
    for name in names:
        agg=SOURCE_AGG/f"{Path(name).stem}_10m.tif"
        if not agg.is_file():raise FileNotFoundError(agg)
        with rasterio.open(agg) as src:
            if src.crs!=crs:raise ValueError(("CRS mismatch",site,src.crs,crs))
            # Full valid UAV coverage, not annotation ROI. Warp validity to 100 m
            # for efficient polygonizing; any 10 m coverage within a cell counts.
            height=math.ceil(src.height/10);width=math.ceil(src.width/10)
            # Binary coverage: 0 remains genuine absence; no float underflow/nodata
            # substitution. Previous warp changed 0 -> 1.4e-45, spuriously
            # marking zero-data pixels valid when testing (>0).
            values=src.read(3,masked=True)
            valid10=(np.ma.filled(values,0)>0).astype(np.uint8)
            valid=np.zeros((height,width),dtype=np.uint8)
            reproject(source=valid10,destination=valid,
                src_transform=src.transform,src_crs=crs,
                dst_transform=src.transform*Affine.scale(10,10),dst_crs=crs,
                resampling=Resampling.max)
            assert valid.any(),("No UAV footprint",agg)
            # Convex hull intentionally fills small holes inside the orthomosaic footprint.
            polys=[shape(geom) for geom,value in shapes(valid,mask=valid.astype(bool),
                transform=src.transform*Affine.scale(10,10)) if value==1]
            hulls.append(unary_union(polys).convex_hull)
    native=unary_union(hulls)
    buffered=native.buffer(BUFFER_DISTANCE_M)
    left,bottom,right,top=buffered.bounds
    ox,oy=standard_origin
    col0=math.floor((left-ox)/SCALE_M);col1=math.ceil((right-ox)/SCALE_M)
    row0=math.floor((oy-top)/SCALE_M);row1=math.ceil((oy-bottom)/SCALE_M)
    width=col1-col0;height=row1-row0
    grid=coarse_transform*Affine.translation(col0,row0)
    assert width>0 and height>0
    feature_year=YEARS[site]
    qwindow=(feature_year-2,feature_year)
    d=dict(site=site,year=feature_year,quarter_years=qwindow,
           crs=crs.to_string(),transform=list(grid)[:6],
           width=width,height=height,pixel_m=SCALE_M,
           footprint_geojson=mapping(native),
           buffered_geojson=mapping(buffered),
           orthomosaics=names)
    REGIONS[site]=d
    (MAP_ROOT/f"09A_{site}_grid.json").write_text(json.dumps(d,indent=2))
    print("FOOTPRINT",site,"ortho area km²",round(native.area/1e6,2),
          "buffered km²",round(buffered.area/1e6,2),
          "raster grid",width,height,"year",feature_year,flush=True)
print("09A done. Outputs:",MAP_ROOT)


In [ ]:
# 09B — Earth Engine: export embeddings + pooled quarterly S2 at 100 m buffered grids
# Run only after 09A. Existing verified exports are reused.
# IMPORTANT: derive each site's original-year satellite predictors, not a mixed-year map.
%pip -q install earthengine-api geemap geedim
import ee,geemap,os,shutil,tempfile,time
from rasterio.warp import transform_bounds
from affine import Affine
EE_PROJECT="musanga-511110"  # same project as original NB04A
REEXPORT_INVALID=False   # False: preserve suspicious existing exports and stop
try:
    ee.Initialize(project=EE_PROJECT)
except Exception:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
S2_REFLECTANCE_BANDS=["B2","B3","B4","B8","B5","B6","B7","B8A","B11","B12"]
SCL_REJECT=[0,1,3,7,8,9,10,11]
EMB_BANDS=[f"A{i:02d}" for i in range(64)]
def s2_masked(img):
    mask=img.select("SCL").remap(SCL_REJECT,[0]*len(SCL_REJECT),1).eq(1)
    return img.select(S2_REFLECTANCE_BANDS).updateMask(mask).multiply(.0001).copyProperties(
        img,["system:time_start"])
def s2_three_year_stack(start,end,region):
    coll=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(region).filterDate(f"{start}-01-01",f"{end+1}-01-01")
          .map(s2_masked))
    chunks=[]
    for q in range(4):
        quarter=coll.filter(ee.Filter.calendarRange(q*3+1,q*3+3,"month"))
        # Existing NB04A uses quarter medians of all three years, not monthly trends.
        med=quarter.median().select(S2_REFLECTANCE_BANDS)
        count=quarter.select("B2").count()
        # Original NB04 samples: quarter reflectance 10->100 m average,
        # clear counts 10->100 m nearest; apply matching resampling operations.
        source_proj=ee.Image(quarter.first()).select("B2").projection()
        averaged=med.setDefaultProjection(source_proj).reduceResolution(
            reducer=ee.Reducer.mean(),maxPixels=1024)
        renamed=[f"Q{q+1}_{b}" for b in S2_REFLECTANCE_BANDS]
        chunks.append(averaged.rename(renamed))
        chunks.append(count.rename(f"Q{q+1}_nclear"))
    return ee.Image.cat(chunks).toFloat()
def s2_single_quarter(start,end,region,q):
    # Same pooled three-year quarter median as original 04A, 11 bands per quarter.
    coll=(ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
          .filterBounds(region).filterDate(f"{start}-01-01",f"{end+1}-01-01")
          .map(s2_masked).filter(ee.Filter.calendarRange(3*q-2,3*q,"month")))
    # Beware original 04A: median at native S2 resolution, then 100m average.
    med=coll.median().select(S2_REFLECTANCE_BANDS)
    native=ee.Image(coll.first()).select("B2").projection()
    med=med.setDefaultProjection(native).reduceResolution(
        reducer=ee.Reducer.mean(),maxPixels=1024)
    count=coll.select("B2").count().rename(f"Q{q}_nclear").toFloat()
    return ee.Image.cat([med.rename([f"Q{q}_{b}" for b in S2_REFLECTANCE_BANDS]),
                        count]).toFloat()
def subset_grid(d,left,top,width,height):
    # Explicit snapped subgrid, never derive geographic transform from EPSG:4326 bbox.
    nd=d.copy()
    nd["width"]=int(width);nd["height"]=int(height)
    nd["transform"]=list(Affine(*d["transform"])*Affine.translation(left,top))[:6]
    return nd
def download_quarter_tiles(img,d,dest,site,q,tile_size=64):
    # Fallback if whole-quarter export exceeds EE per-request memory.
    # Each 64x64 tile is aligned to the SAME projected pixel lattice.
    import rasterio
    from rasterio.windows import Window
    from rasterio.crs import CRS
    dest.parent.mkdir(parents=True,exist_ok=True)
    partial=dest.with_suffix(".partial.tif")
    nodata=-9999.
    profile=dict(driver="GTiff",height=d["height"],width=d["width"],
        count=11,crs=CRS.from_string(d["crs"]),transform=Affine(*d["transform"]),
        dtype="float32",nodata=nodata,compress="deflate",tiled=True)
    with rasterio.open(partial,"w",**profile) as out:
        for top in range(0,d["height"],tile_size):
            for left in range(0,d["width"],tile_size):
                h=min(tile_size,d["height"]-top)
                w=min(tile_size,d["width"]-left)
                tile_grid=subset_grid(d,left,top,w,h)
                tile_cache=MAP_ROOT/f"09B_parts_{site}_Q{q}"
                tile_cache.mkdir(exist_ok=True)
                tile=tile_cache/f"r{top:04d}_c{left:04d}.tif"
                if tile.is_file():
                    verify_sat_export(tile,tile_grid,11,min_valid=0)
                else:
                    print("QUARTER TILE",site,q,top,left,"size",h,w,flush=True)
                    download_sat_image(img,tile_grid,tile,min_valid=0)
                with rasterio.open(tile) as src:
                    out.write(src.read(),window=Window(left,top,w,h))
    verify_sat_export(partial,d,11,min_valid=100)
    partial.replace(dest)
def make_quarter_stack(start,end,region,d,site):
    # Restartable by quarter. Existing successful parts stay on Drive.
    paths=[]
    for q in range(1,5):
        piece=MAP_ROOT/f"09B_{site}_S2_Q{q}_{SCALE_M}m.tif"
        if piece.is_file():
            print("CACHED QUARTER",site,q,
                  verify_sat_export(piece,d,11),flush=True)
        else:
            img=s2_single_quarter(start,end,region,q)
            try:
                print("QUARTER EXPORT",site,q,"11 bands",flush=True)
                download_sat_image(img,d,piece)
            except Exception as exc:
                if "memory limit" not in str(exc).lower() and "400" not in str(exc):
                    raise
                print("EE quarter request too large; retrying as 64x64 snapped tiles",
                      site,q,type(exc).__name__,flush=True)
                download_quarter_tiles(img,d,piece,site,q)
        paths.append(piece)
    final=MAP_ROOT/f"09B_{site}_S2_QUARTERLY_3YR_{SCALE_M}m.tif"
    from rasterio.crs import CRS
    profile=dict(driver="GTiff",width=d["width"],height=d["height"],
                 count=44,crs=CRS.from_string(d["crs"]),
                 transform=Affine(*d["transform"]),
                 dtype="float32",nodata=-9999.,compress="deflate",tiled=True)
    temp=final.with_suffix(".partial.tif")
    with rasterio.open(temp,"w",**profile) as out:
        for q,p in enumerate(paths,1):
            with rasterio.open(p) as src:
                out.write(src.read(),indexes=list(range((q-1)*11+1,q*11+1)))
    verify_sat_export(temp,d,44)
    temp.replace(final)
    print("COMBINED 44 BANDS",final,flush=True)

def embedding_stack(year,region):
    imgs=(ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filterDate(f"{year}-01-01",f"{year+1}-01-01").filterBounds(region))
    n=imgs.size().getInfo()
    assert 0<n<300,("Suspicious embedding tile count: check WGS84 geographic geometry",year,n)
    print("Embedding tiles",year,n,flush=True)
    source_proj=ee.Image(imgs.first()).select("A00").projection()
    return imgs.mosaic().select(EMB_BANDS).setDefaultProjection(
        source_proj).reduceResolution(reducer=ee.Reducer.mean(),maxPixels=1024).toFloat()
def verify_sat_export(path,d,n_bands,min_valid=100):
    with rasterio.open(path) as ds:
        assert ds.count==n_bands,(path,"bands",ds.count,n_bands)
        assert ds.crs==rasterio.crs.CRS.from_string(d["crs"]),(path,"CRS")
        assert ds.width==d["width"] and ds.height==d["height"],(path,"shape")
        expected=Affine(*d["transform"])
        assert ds.transform.almost_equals(expected,precision=1e-5),(
            path,"wrong grid",ds.transform,expected)
        assert ds.nodata is not None,(path,"explicit nodata missing")
        preview=ds.read(1,masked=True)
        n_valid=int(np.ma.count(preview))
        assert n_valid>=min_valid,(path,"Too few valid pixels",n_valid,min_valid)
    return n_valid
def download_sat_image(image,d,dest,min_valid=100):
    from rasterio.warp import reproject,Resampling
    from rasterio.crs import CRS
    with tempfile.TemporaryDirectory(dir="/content",prefix="musanga09_") as temp:
        raw=Path(temp)/"earth_engine_raw.tif"
        fixed=Path(temp)/"aligned.tif"
        tf=Affine(*d["transform"]);crs=CRS.from_string(d["crs"])
        x0,y0=tf*(0,0);x1,y1=tf*(d["width"],d["height"])
        bounds=(min(x0,x1),min(y0,y1),max(x0,x1),max(y0,y1))
        west,south,east,north=transform_bounds(crs,"EPSG:4326",*bounds,densify_pts=21)
        region=ee.Geometry.Rectangle([west,south,east,north],proj="EPSG:4326",
                                       geodesic=False)
        print("EE DOWNLOAD",dest.name,"bands",image.bandNames().size().getInfo(),
              "grid",d["width"],d["height"],flush=True)
        geemap.download_ee_image(image,filename=str(raw),region=region,
            crs=d["crs"],crs_transform=d["transform"],
            shape=(d["height"],d["width"]))
        with rasterio.open(raw) as src:
            assert src.count==image.bandNames().size().getInfo()
            valid_raw=int(np.ma.count(src.read(1,masked=True)))
            print("RAW RASTER QA",dest.name,"native CRS",str(src.crs),
                  "grid",src.shape,"valid band1 pixels",valid_raw,flush=True)
            assert valid_raw>=min_valid,("Earth Engine returned empty imagery; check geographic bounds, year and source masks",dest.name,valid_raw,min_valid)
            same=src.crs==crs and src.width==d["width"] and src.height==d["height"] and (
                src.transform.almost_equals(tf,precision=1e-5))
            if not same:
                # Do not accept a shifted or swapped export: only known subpixel
                # geedim alignment variation, then explicitly reproject to target.
                pixel_error=max(abs(src.transform.c-tf.c)/SCALE_M,
                                abs(src.transform.f-tf.f)/SCALE_M)
                assert src.crs==crs and src.shape==(d["height"],d["width"]) and pixel_error<=.5,(
                    "Unexpected Earth Engine grid or swapped dimensions",src.shape,
                    src.transform,tf,pixel_error)
                profile=src.profile.copy()
                profile.update(transform=tf,crs=crs,nodata=-9999.,compress="deflate")
                with rasterio.open(fixed,"w",**profile) as dst:
                    for b in range(1,src.count+1):
                        destband=np.full(src.shape,-9999.,np.float32)
                        reproject(rasterio.band(src,b),destband,
                            src_nodata=src.nodata,dst_nodata=-9999.,
                            src_transform=src.transform,src_crs=src.crs,
                            dst_transform=tf,dst_crs=crs,resampling=Resampling.nearest)
                        dst.write(destband,b)
            else:
                profile=src.profile.copy()
                profile.update(nodata=-9999.,compress="deflate")
                with rasterio.open(fixed,"w",**profile) as dst:
                    for b in range(1,src.count+1):
                        a=src.read(b).astype(np.float32)
                        if src.nodata is not None:a[a==src.nodata]=-9999.
                        dst.write(a,b)
        verify_sat_export(fixed,d,image.bandNames().size().getInfo(),min_valid=min_valid)
        shutil.copy2(fixed,dest)
for site,d in REGIONS.items():
    year=d["year"];start,end=d["quarter_years"]
    # The buffered GeoJSON stores projected coordinates (metres), not lon/lat.
    # ee.Geometry(geojson) silently treats them as longitude/latitude; that
    # returned ~6973 unrelated embedding tiles and an empty raster.
    tf=Affine(*d["transform"])
    x0,y0=tf*(0,0);x1,y1=tf*(d["width"],d["height"])
    west,south,east,north=transform_bounds(d["crs"],"EPSG:4326",
        min(x0,x1),min(y0,y1),max(x0,x1),max(y0,y1),densify_pts=21)
    assert -180<=west<east<=180 and -90<=south<north<=90,(
        "Invalid geographic satellite footprint",site,(west,south,east,north))
    georef=ee.Geometry.Rectangle([west,south,east,north],
             proj="EPSG:4326",geodesic=False)
    print("EE geographic bounds",site,
          tuple(round(v,5) for v in (west,south,east,north)),flush=True)
    for kind in ("EMB_ANNUAL","S2_QUARTERLY_3YR"):
        dest=MAP_ROOT/f"09B_{site}_{kind}_{SCALE_M}m.tif"
        expected=64 if kind=="EMB_ANNUAL" else 44
        if dest.is_file():
            try:
                print("VERIFIED SAT CACHE",dest.name,
                    verify_sat_export(dest,d,expected),flush=True)
                continue
            except Exception as err:
                if not REEXPORT_INVALID:raise RuntimeError(
                    f"Invalid cached raster preserved: {dest}; {err}") from err
                dest.rename(dest.with_suffix(".invalid.tif"))
        if kind=="S2_QUARTERLY_3YR":
            make_quarter_stack(start,end,georef,d,site)
            continue
        img=embedding_stack(year,georef)
        download_sat_image(img,d,dest)
        print("SAVED",dest,flush=True)
print("09B finished. Run 09C to compare predictors at UAV training cells BEFORE applying model.")


In [ ]:
# 09C — Audit satellite predictor compatibility, then predict over full buffered AOI
# Requires Sections 06, 07 (weighted calibration), 09A and 09B.
# Output bands: predicted fraction, nominal lower/upper 90%, ensemble SD,
# PCA predictor-distance and binary outside-training-domain flag.
from rasterio.features import geometry_mask
from rasterio.transform import rowcol
from rasterio.windows import Window
from rasterio.crs import CRS
from affine import Affine
import gc
assert "mapping_cal" in globals(),"Run Section 07 (including weighted calibration) first"
assert SCALE_M==100,"Regional export designed for a 100 m target grid."
EMBEDDING_COLUMNS=[f"EMB_ANNUAL__{i:02d}" for i in range(1,65)]
QUARTERLY_COLUMNS=[f"S2_QUARTERLY_3YR__{i:02d}" for i in range(1,45)]
expected_columns=EMBEDDING_COLUMNS+QUARTERLY_COLUMNS
assert FEATURE_COLS==expected_columns,(
    "Feature order mismatch: must match training",FEATURE_COLS[:5],len(FEATURE_COLS))
MAP_COLUMNS=["predicted","lower90","upper90","ensemble_sd",
             "novelty_distance","outside_domain"]
# Spatial-fold median of interval corrections, carried to the full-data refit.
# This is provisional uncertainty only: no confirmed geographic coverage guarantee.
interval_map={"bins":[]}
for i in range(len(mapping_cal[0]["calibration"]["bins"])):
    template=mapping_cal[0]["calibration"]["bins"][i]
    vals=np.array([m["calibration"]["bins"][i]["limits"] for m in mapping_cal],float)
    limits=np.median(vals,axis=0)
    interval_map["bins"].append(dict(min_pred=template["min_pred"],
                   max_pred=template["max_pred"],limits=limits.tolist()))
# Freeze the full-data model and novelty metric once; do not refit per forest.
final_models,_=fit_ensemble(df,[],SEED+901)
novel_train=balanced_training(df,np.random.default_rng(SEED+900))
xtrain=novel_train[FEATURE_COLS].to_numpy(np.float32)
scaler=StandardScaler().fit(xtrain)
ztrain=scaler.transform(xtrain)
reduction=PCA(n_components=min(10,ztrain.shape[1],len(ztrain)-1),
              random_state=SEED).fit(ztrain)
ztrain=reduction.transform(ztrain)
knn=NearestNeighbors(n_neighbors=2).fit(ztrain)
training_nearest=knn.kneighbors(ztrain)[0][:,1]
novel_limit=float(np.quantile(training_nearest,.95))
def feature_audit(site,d):
    # Transform coordinates from the OLD annual grid to the expanded buffered grid.
    oldpath=SAT/f"{site}_RAW_ANNUAL_{YEARS[site]}_10m.tif"
    test=df[df.site==site]
    sample=test.sample(n=min(400,len(test)),random_state=SEED)
    with rasterio.open(oldpath) as old:
        original=old.transform*Affine.scale(SCALE_M/10,SCALE_M/10)
    xx,yy=original*(sample.col.to_numpy()+.5,sample.row.to_numpy()+.5)
    inv=~Affine(*d["transform"])
    buffer_cols,buffer_rows=inv*(xx,yy)
    cc=np.floor(buffer_cols).astype(int);rr=np.floor(buffer_rows).astype(int)
    valid=(rr>=0)&(cc>=0)&(rr<d["height"])&(cc<d["width"])
    if not valid.all():raise ValueError((site,"Cached UAV cells outside 5 km grid",
                                        int(np.sum(~valid))))
    deviations={}
    new_values=[]
    for kind,cols in [("EMB_ANNUAL",EMBEDDING_COLUMNS),
                      ("S2_QUARTERLY_3YR",QUARTERLY_COLUMNS)]:
        filepath=MAP_ROOT/f"09B_{site}_{kind}_{SCALE_M}m.tif"
        assert filepath.is_file(),filepath
        with rasterio.open(filepath) as img:
            vals=np.column_stack([img.read(b)[rr,cc] for b in range(1,img.count+1)])
        actual=sample[cols].to_numpy(float)
        # Exclude clear-count bands from spectral compatibility metrics:
        # previous pipeline used NN over 10 m grid for count bands.
        checked=np.array([i for i in range(len(cols))
            if not (kind=="S2_QUARTERLY_3YR" and (i+1)%11==0)])
        assert np.isfinite(vals).all() and (vals>-9998).all(),(
            site,kind,"Missing buffered predictors over original UAV cells")
        error=np.abs(actual[:,checked]-vals[:,checked])
        deviations[kind]=dict(n_cells=len(sample),n_features=len(checked),
             median_absolute_error=float(np.median(error)),
             p95_absolute_error=float(np.quantile(error,.95)),
             mean_absolute_error=float(error.mean()))
        # Sanity tolerance: tight median mismatch indicates wrong feature production.
        threshold=.035 if kind=="EMB_ANNUAL" else .025
        if np.median(error)>threshold:
            raise ValueError((site,kind,"Exported predictors disagree with original cached"
                    " satellite features. STOP before regional prediction.",deviations[kind]))
        new_values.append(vals)
    return deviations
audit=[]
for site,d in REGIONS.items():
    record=dict(site=site,**feature_audit(site,d))
    audit.append(record)
    print("SATELLITE AUDIT",record,flush=True)
(MAP_ROOT/"09C_feature_compatibility_audit.json").write_text(json.dumps(audit,indent=2))
print("All buffered predictors passed training-cache compatibility audit.",flush=True)
def predict_chunk(values):
    # values shape=(n_valid,108): 64 embeddings + 44 quarterly bands
    x=np.asarray(values,np.float32)
    preds=np.stack([np.clip(m.predict(x),0,1) for m in final_models],axis=0)
    pred=preds.mean(axis=0);sd=preds.std(axis=0)
    lo,hi=interval_bounds(pred,interval_map)
    z=reduction.transform(scaler.transform(x))
    dist=knn.kneighbors(z,n_neighbors=1)[0][:,0]
    return np.stack([pred,lo,hi,sd,dist,(dist>novel_limit).astype(float)],axis=0)
map_log=[]
for site,d in REGIONS.items():
    outfile=MAP_ROOT/f"09C_{site}_buffer5km_{SCALE_M}m_prediction_uncertainty.tif"
    year=d["year"]
    prof=dict(driver="GTiff",height=d["height"],width=d["width"],
        count=6,crs=CRS.from_string(d["crs"]),
        transform=Affine(*d["transform"]),dtype="float32",
        nodata=-9999.,compress="deflate",tiled=True,
        blockxsize=256,blockysize=256,BIGTIFF="IF_SAFER")
    buffered_geom=shape(d["buffered_geojson"])
    # Read by geographic windows, not huge 108-band raster in memory.
    with rasterio.open(MAP_ROOT/f"09B_{site}_EMB_ANNUAL_{SCALE_M}m.tif") as emb, \
         rasterio.open(MAP_ROOT/f"09B_{site}_S2_QUARTERLY_3YR_{SCALE_M}m.tif") as qtr:
        assert emb.count==64 and qtr.count==44
        assert emb.transform.almost_equals(prof["transform"],precision=1e-5)
        assert qtr.transform.almost_equals(prof["transform"],precision=1e-5)
        if outfile.is_file():
            with rasterio.open(outfile) as check:
                if check.count==6 and check.shape==(d["height"],d["width"]) and \
                    check.transform.almost_equals(prof["transform"],precision=1e-5):
                    print("MAP EXISTS — preserving",outfile,flush=True)
                    continue
            raise RuntimeError("Existing map has unexpected geometry; remove manually: "+str(outfile))
        temp=outfile.with_suffix(".partial.tif")
        valid_total=0;inside_ortho_total=0
        with rasterio.open(temp,"w",**prof) as dst:
            for i,label in enumerate(MAP_COLUMNS,1):dst.set_band_description(i,label)
            # Coarse 256x256 blocks; memory bounded.
            for top in range(0,d["height"],256):
                for left in range(0,d["width"],256):
                    h=min(256,d["height"]-top);w=min(256,d["width"]-left)
                    window=Window(left,top,w,h)
                    e=emb.read(window=window).reshape(64,-1).T
                    q=qtr.read(window=window).reshape(44,-1).T
                    mask=(np.isfinite(e).all(axis=1)&np.isfinite(q).all(axis=1)&
                        (e>-9998).all(axis=1)&(q>-9998).all(axis=1))
                    gmask=geometry_mask([buffered_geom],out_shape=(h,w),
                        transform=rasterio.windows.transform(window,prof["transform"]),
                        invert=True).ravel()
                    mask&=gmask
                    out=np.full((6,h*w),-9999.,np.float32)
                    if mask.any():
                        values=np.concatenate([e[mask],q[mask]],axis=1)
                        out[:,mask]=predict_chunk(values)
                        valid_total+=int(mask.sum())
                    dst.write(out.reshape(6,h,w),window=window)
            dst.update_tags(
                description="Musanga UAV-reference satellite abundance prediction",
                reference_year=str(year),annual_embedding_year=str(year),
                seasonal_S2_years=f"{year-2}-{year}",
                uncertainty_warning="Nominal 90% bounds; high-cover undercoverage and domain shift",
                AOA_method="Exploratory PCA standardized feature nearest-neighbor threshold",
                authorship="Musanga NB04 09C, 5km buffered valid orthomosaic footprint")
        temp.replace(outfile)
    summary=dict(site=site,year=year,file=str(outfile),n_predicted=int(valid_total),
                 buffer_m=BUFFER_DISTANCE_M,scale_m=SCALE_M)
    map_log.append(summary)
    print("MAP READY",summary,flush=True)
manifest=dict(scale_m=SCALE_M,buffer_m=BUFFER_DISTANCE_M,
    prediction_feature_order=expected_columns,
    source_satellite_years=YEARS,models="5-member stratified HistGradientBoostingRegressor",
    training_forest_cap=TRAIN_CAP_PER_FOREST,
    calibration="spatial fold weighted forest balanced residual quantiles",
    calibration_limits=interval_map,
    novelty_95_percent_threshold=novel_limit,
    bands=MAP_COLUMNS,
    WARNING=("Prediction bounds are provisional, not validated for geographic extrapolation; "
             "observed >=40% abundance conditional interval coverage was ~58%. "
             "Site-specific reference years differ."))
(MAP_ROOT/"09C_mapping_manifest.json").write_text(json.dumps(manifest,indent=2))
pd.DataFrame(map_log).to_csv(MAP_ROOT/"09C_map_inventory.csv",index=False)
print("Finished buffered 5 km per-site satellite mapping:",MAP_ROOT,flush=True)
